# 恢復本聖經＋生命讀經：下載到 Google Drive

在 Google 的雲端主機上執行下載程式，檔案直接存進你的 Google Drive，不會用到你的電腦或公司網路。完成後在家裡用自己的電腦從 Drive 下載或同步即可。

**步驟**：由上而下逐格按 ▶ 執行。第 2 格會要求允許存取 Google Drive，請選自己的帳號並允許。

- 全部約 27GB（經文 1,189 章、生命讀經 1,984 個不重複音訊），預計需要數小時。執行期間請保持這個分頁開著。
- **中斷沒關係**：如果 Colab 斷線或逾時，重新整理頁面後再從第 1 格依序執行，會從斷點繼續，已完成的檔案不會重下。
- 內容版權屬臺灣福音書房，僅供個人離線研讀，不公開、不轉散布。


In [ ]:
# 第 1 格：設定（通常不用改）
BOOKS = '1-66'                        # 例如 '1-5,40' 只抓部分書卷
COLLECTIONS = 'bible,life-study'      # 'bible'＝恢復本、'life-study'＝生命讀經
DOWNLOAD_AUDIO = True                 # False＝只抓文字與原始網頁（約 100MB，幾十分鐘）
DRIVE_FOLDER = 'twgbr-archive'        # 存在「我的雲端硬碟」下的資料夾名稱
MAKE_ZIP = False                      # 全部完成後是否另外打包 zip（會多佔約同樣大小的空間）


In [ ]:
# 第 2 格：連接 Google Drive
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = Path('/content/drive/MyDrive')/DRIVE_FOLDER
OUTPUT.mkdir(parents=True, exist_ok=True)
print('資料保存位置：我的雲端硬碟/'+DRIVE_FOLDER)


In [ ]:
# 第 3 格：取得固定版本的下載程式並核對 SHA-256
import hashlib, subprocess, sys, urllib.request
COMMIT = '39962158f7306b3240974ca859a60dba47fa4625'
EXPECTED_SHA256 = '888aa7a0a7dd3a5bdfd1f51c649d8fff338a6b6036babfb941f67544bdbdaf84'
URL = f'https://raw.githubusercontent.com/timo893131/Meeting-Live-Captions/{COMMIT}/scripts/download_twgbr_archive.py'
SCRIPT = Path('/content/download_twgbr_archive.py')
data = urllib.request.urlopen(URL, timeout=60).read()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError('下載程式的 SHA-256 不符，已停止。請不要執行未核對的程式。')
SCRIPT.write_bytes(data)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'beautifulsoup4'], check=True)
print('程式版本', COMMIT[:12], '已核對')


In [ ]:
# 第 4 格：開始下載（每卷顯示一行進度；可隨時中斷，重跑會續抓）
command = [sys.executable, '-u', str(SCRIPT), '--output', str(OUTPUT), '--books', BOOKS,
           '--collections', COLLECTIONS, '--quiet']
if not DOWNLOAD_AUDIO:
    command.append('--no-audio')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='', flush=True)
code = process.wait()
print('\n結束代碼：', code, '（0＝全部成功；1＝有失敗項目，重新執行這格即可補抓）')


In [ ]:
# 第 5 格：檢查完成狀況，並確保檔案都已寫入 Google Drive
import json, re
readme = (OUTPUT/'README.md').read_text(encoding='utf-8')
print(readme[readme.index('## 總數'):readme.index('## 逐卷清單')])
sys.path.insert(0, str(SCRIPT.parent)); import download_twgbr_archive as archive
wanted = {archive.folder(n) for n in archive.parse_books(BOOKS)}
bible_on, life_on = 'bible' in COLLECTIONS, 'life-study' in COLLECTIONS
problems = []
for line in readme.splitlines():
    m = re.match(r'\| (\d\d-\S+) \| (\S) \| (\d+)／(\S+) \| (\d+)／(\S+) \| (\d+) \|', line)
    if not m or m[1] not in wanted:
        continue
    bible_bad = bible_on and (m[2] != '✓' or (DOWNLOAD_AUDIO and m[3] != m[4]))
    life_bad = life_on and (m[5] != m[6] or (DOWNLOAD_AUDIO and m[7] != m[6]))
    if bible_bad or life_bad:
        problems.append(m[1])
errors = [l for l in (OUTPUT/'errors.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
if problems or errors:
    print('尚未完成的書卷：', '、'.join(problems) or '無', '｜失敗項目：', len(errors))
    print('請重新執行第 4 格補抓，再執行這一格。')
else:
    print('全部書卷完成，沒有失敗項目。')
    if MAKE_ZIP:
        print('壓縮檔：', archive.make_zip(OUTPUT, OUTPUT.parent/(DRIVE_FOLDER+'.zip')))
drive.flush_and_unmount()   # 等待所有檔案上傳到 Google Drive
print('已同步到 Google Drive。可以在家用自己的電腦開啟「我的雲端硬碟/'+DRIVE_FOLDER+'」下載。')
